# Intermediate Workshop: Regression, NLP, PCA, & AI Loops

This notebook contains the exercises for the intermediate workshop. Fill in the code where indicated by `# YOUR CODE HERE`.

# Section 1: Regression

#### Data Source
The regression dataset comes from the V-Dem (Varieties of Democracy) project and the IMF WEO database. The original file is cached locally at `../../../data/examples/week_10/democracy_gdp.csv` (relative to the project root directory).

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Load and clean data
df_reg = pd.read_csv("../../../data/examples/week_10/democracy_gdp.csv")[['country', 'year', 'NGDPDPC', 'v2x_freexp']]
df_reg = df_reg.dropna(subset=['NGDPDPC'])
df_reg['NGDPDPC_log'] = np.log(df_reg['NGDPDPC'])
print("Data loaded, shape:", df_reg.shape)

Data loaded, shape: (4515, 5)


In [2]:
# Fit OLS model directly
X = sm.add_constant(df_reg['v2x_freexp'])
model = sm.OLS(df_reg['NGDPDPC_log'], X).fit()
print("OLS Parameters:")
print(model.params)
print("R-squared:", model.rsquared)

OLS Parameters:
const         8.631178
v2x_freexp    1.378770
dtype: float64
R-squared: 0.0859597552275434


In [3]:
# Create OLS method with input args
def run_custom_ols(data, y_col, x_col):
    X = sm.add_constant(data[x_col])
    model = sm.OLS(data[y_col], X).fit()
    return model.params

print("Custom OLS Parameters:")
print(run_custom_ols(df_reg, 'NGDPDPC_log', 'v2x_freexp'))

Custom OLS Parameters:
const         8.631178
v2x_freexp    1.378770
dtype: float64


# Section 2: Text Data (NLP)

#### Data Source
The text data is processed from a real IMF country report cached locally at `../../../data/examples/week_7/imf_reports/Angola_2026.pdf`.

In [4]:
from PyPDF2 import PdfReader

# Read PDF text page-by-page
def read_pdf(file_path):
    reader = PdfReader(file_path)
    result = {}
    for i, page in enumerate(reader.pages):
        result[f"Page {i+1}"] = page.extract_text()
    return result

pdf_imf = read_pdf("../../../data/examples/week_7/imf_reports/Angola_2026.pdf")
page_text = pdf_imf["Page 5"]
print("Extracted page text snippet:")
print(page_text[:200])

Extracted page text snippet:
 
 
ANGOLA 
STAFF REPORT FOR THE 202 6 ARTICLE IV CONSULTATION  
 
KEY ISSUES  
Context . A significant decline  in oil production weakened fiscal and external positions. 
Overall growth held up in 20


In [5]:
import spacy

# Load spaCy pipeline and find verbs
try:
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(page_text[:1000])
    verbs = []
    for token in doc:
        if token.pos_ == "VERB":
            verbs.append(token.text)
    print("Found verbs (first 10):", verbs[:10])
except Exception as e:
    print("spaCy model error:", e)

Found verbs (first 10): ['weakened', 'held', 'supported', 'eased', 'remained', 'remain', 'constrained', 'declined', 'improved', 'levated']


# Section 3: Machine Learning (PCA)

#### Data Source
The dataset for index creation comes from PISA (Programme for International Student Assessment) and World Bank education statistics. The original file is cached locally at `../../../data/examples/week_10/education.csv` (relative to the project root directory).

In [6]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Load data
df_edu = pd.read_csv("../../../data/examples/week_10/education.csv").dropna()
data = df_edu[['GINI', 'LibDem', 'Public Spending on Education', 'Private Spending on Education']]
print("Data loaded, shape:", data.shape)

Data loaded, shape: (122, 4)


In [7]:
# Scale and run PCA
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)
pca = PCA(n_components=1)
pca_result = pca.fit_transform(scaled_data)
print("PCA Explained Variance:", pca.explained_variance_ratio_)
print("First 2 PCA values:", pca_result[:2].flatten())

PCA Explained Variance: [0.67408271]
First 2 PCA values: [-2.12284022  1.79364941]


# Section 4: AI Loops

#### Data Source
The inputs are a simulated set of page strings representing excerpts from International Monetary Fund (IMF) country reports.

In [8]:
import requests

OLLAMA_MODEL = "gemma3:12b"

# Helper function to query local Ollama LLM
def llm_ollama(prompt, model=OLLAMA_MODEL):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    return response.json()["response"]

# Loop over pages to generate summaries
pages = [
    "Inflation has eased from its peak.",
    "Angola's economic outlook remains positive but vulnerable.",
    "Private credit growth has declined steadily."
]
summaries = []
for page in pages:
    summary = llm_ollama(f"Summarize the following text in one sentence: {page}")
    summaries.append(summary)

print("Ollama Summaries:")
print(summaries)

Ollama Summaries:
['Inflation, while still present, has decreased since reaching its highest point.\n', "Angola's economy is expected to perform well, but its progress is susceptible to potential risks and challenges.", "Private credit growth is experiencing a consistent downward trend.\n\n\n\nLet me know if you'd like another summary!"]


In [9]:
# Run the custom NumPy OLS regression pipeline built in supernova assignment
import sys
sys.path.insert(0, "../../../src/workshops/regression_lib")
from pipeline import RegressionPipeline
RegressionPipeline().run()

=== OLS Regression Results ===
Intercept (B0): 2.5865
Slope     (B1): 2.1355
R-squared:      0.5906

Plot saved to reports/viz/regression_pipeline.png


array([2.58649448, 2.13550335])